First of all, import the modules needed.

In [23]:
import zipfile
import requests
import os
import optuna
import time
import shutil

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import date, timedelta
from numba import njit
from pathlib import Path

Then, download and save the data.

In [ ]:
symbol = "BTCUSDT"
interval = "1m"
base_url = "https://data.binance.vision/data/spot/daily/klines"

start = date(2023, 1, 1)   # inclusive
end = date(2026, 2, 28)   # inclusive

download_dir = "tmp_data"
os.makedirs(download_dir, exist_ok=True)

# 1) Download and unzip daily files
d = start
while d <= end:
    ds = d.strftime("%Y-%m-%d")
    zip_name = f"{symbol}-{interval}-{ds}.zip"
    url = f"{base_url}/{symbol}/{interval}/{zip_name}"
    zip_path = os.path.join(download_dir, zip_name)

    print(f"Fetching {url}")
    r = requests.get(url)
    if r.status_code == 200:
        with open(zip_path, "wb") as f:
            f.write(r.content)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(download_dir)
        os.remove(zip_path)  # keep only CSV
        print(f"OK {ds}")
    else:
        print(f"Missing or error for {ds}: {r.status_code}")
    d += timedelta(days=1)

# 2) Concatenate all CSVs in time order
csv_files = sorted(f for f in os.listdir(download_dir) if f.endswith(".csv") and f.startswith(f"{symbol}-{interval}-"))

dfs = []
for fname in csv_files:
    path = os.path.join(download_dir, fname)
    df = pd.read_csv(path, header=None)
    # Binance kline schema:
    # 0 open time, 1 open, 2 high, 3 low, 4 close, 5 volume,
    # 6 close time, 7 quote asset volume, 8 number of trades,
    # 9 taker buy base volume, 10 taker buy quote volume, 11 ignore
    dfs.append(df)

if not dfs:
    raise RuntimeError("No CSVs downloaded; check dates and URLs.")

full = pd.concat(dfs, ignore_index=True)

# Optional: sort by open time and drop duplicates in case of overlap
full = full.sort_values(0).drop_duplicates(subset=0)

# 3) Save a single merged CSV
out_file = f"{symbol}-{interval}-{start.strftime('%Y-%m-%d')}_{end.strftime('%Y-%m-%d')}.csv"
full.to_csv(out_file, index=False, header=False)
print("Merged CSV written to:", out_file)

# Clear all the temp .zip files
root = Path(download_dir)
for item in root.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

Define the backtest functions:

In [24]:
@njit(fastmath=True, cache=True)
def simulate(closes: np.ndarray, signal: np.ndarray, sl_rate: float, tp_rate: float,
             initial_capital: float, fee_rate: float):
    n = len(closes)
    equity = np.empty(n, dtype=np.float64)
    cash = initial_capital
    position = 0.0
    entry_price = 0.0
    total_traded = 0.0
    max_lev = 0.0
    trades = 0

    for i in range(n):
        close_p = closes[i]

        # EXIT (SL or TP)
        if position > 0.0:
            if close_p <= entry_price * (1.0 - sl_rate) or close_p >= entry_price * (1.0 + tp_rate):
                sell_notional = position * close_p
                total_traded += sell_notional
                cash += sell_notional * (1.0 - fee_rate) if fee_rate > 0.0 else sell_notional
                position = 0.0
                entry_price = 0.0

        # ENTRY
        if position == 0.0 and signal[i] and cash > 0.01:
            notional = cash / (1.0 + fee_rate) if fee_rate > 0.0 else cash
            size = notional / close_p
            position = size
            cash = 0.0
            total_traded += notional
            entry_price = close_p
            trades += 1

        equity[i] = cash + position * close_p

        if equity[i] > 0.0 and position > 0.0:
            lev = (position * close_p) / equity[i]
            if lev > max_lev:
                max_lev = lev

    return equity, total_traded, max_lev, trades


def compute_metrics(equity: np.ndarray, initial_capital: float, total_traded: float, n: int):
    final_pnl = equity[-1] - initial_capital
    if n < 2:
        return {'final_pnl': final_pnl, 'sharpe': 0.0, 'ann_sharpe': 0.0,
                'ann_turnover': 0.0, 'max_dd_pct': 0.0}

    returns = np.diff(equity) / equity[:-1]
    sharpe = 0.0
    ann_sharpe = 0.0
    if len(returns) > 30 and np.std(returns) > 1e-8:
        sharpe = np.mean(returns) / np.std(returns)
        annual_factor = 365.25 * 24 * 60
        ann_sharpe = sharpe * np.sqrt(annual_factor)

    cummax = np.maximum.accumulate(equity)
    dd = (equity - cummax) / cummax
    max_dd_pct = -dd.min() * 100 if dd.min() < 0 else 0.0

    years = n / (365.25 * 24 * 60)
    ann_turnover = (total_traded / initial_capital) / years if years > 0 else 0.0

    return {
        'final_pnl': final_pnl,
        'sharpe': sharpe,
        'ann_sharpe': ann_sharpe,
        'ann_turnover': ann_turnover,
        'max_dd_pct': max_dd_pct
    }

Then, run the backtest (specifically for $(x, y) = (10, 3)$):

In [ ]:
# ====================== MAIN ======================
print("Loading CSV...")
data = pd.read_csv("data/BTCUSDT-1m-2023-01-01_2026-02-28.csv", usecols=[i for i in range(6)], names=["OpenTimestamp[ms]", "Open", "High", "Low", "Close", "Volume"])  # Ignore the last column
threshold = 1e14  # Threshold for ms
data['OpenTimestamp[ms]'] = np.where(data['OpenTimestamp[ms]'] > threshold, data['OpenTimestamp[ms]'] // 1000, data['OpenTimestamp[ms]'])  # Convert all time data in time column to ms
df = data
df['datetime'] = pd.to_datetime(df["OpenTimestamp[ms]"], unit="ms")
df = df.sort_values("datetime").reset_index(drop=True)
n = len(df)
print(f"Data loaded: {n:,} minutes (~{n / (60 * 24):.1f} days)")

closes = df['Close'].values
lows = df['Low'].values
initial_capital = 1_000_000.0

# ====================== SINGLE BACKTEST ======================
x, sl, tp = 10, 0.002, 0.004
print(f"\nRunning default backtest (x={x}, SL={sl * 100:.1f}%, TP={tp * 100:.1f}%)...")

past_min = df['Low'].shift(1).rolling(window=x, min_periods=x).min().values
signal_def = (lows < past_min) & (~np.isnan(past_min))

equity_before, traded_before, _, trades_before = simulate(closes, signal_def, sl, tp, initial_capital, 0.0)
metrics_before = compute_metrics(equity_before, initial_capital, traded_before, n)

equity_after, traded_after, max_lev, trades_after = simulate(closes, signal_def, sl, tp, initial_capital, 0.001)
metrics_after = compute_metrics(equity_after, initial_capital, traded_after, n)

print("\n" + "=" * 80)
print(f"SINGLE RESULTS (x={x}, SL={sl * 100:.1f}%, TP={tp * 100:.1f}%)")
print("=" * 80)
print(f"Final PnL BEFORE : ${metrics_before['final_pnl']:,.2f}")
print(f"Final PnL AFTER  : ${metrics_after['final_pnl']:,.2f}")
print(f"Sharpe BEFORE    : {metrics_before['sharpe']:.4f}")
print(f"Sharpe AFTER     : {metrics_after['sharpe']:.4f}")
print(f"Ann. Sharpe BEFORE: {metrics_before['ann_sharpe']:.2f}")
print(f"Ann. Sharpe AFTER: {metrics_after['ann_sharpe']:.2f}")
print(f"Max DD BEFORE    : {metrics_before['max_dd_pct']:.2f}%")
print(f"Max DD AFTER     : {metrics_after['max_dd_pct']:.2f}%")
print(f"Total trades     : {trades_after:,}")
print(f"Ann. Turnover    : {metrics_after['ann_turnover']:.2f}x")
print(f"Max Leverage     : {max_lev:.2f}x")
print("=" * 80)

# Save equity curve for Flask dashboard
equity_df = pd.DataFrame({
    'datetime': df['datetime'],
    'pnl_before': equity_before - initial_capital,
    'pnl_after': equity_after - initial_capital
})
equity_df.to_csv('equity_curve.csv', index=False)
print("Saved 'equity_curve.csv' for the Flask dashboard.")

# Cumulative PnL plot (static)
plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_before - initial_capital, label='Before 0.1% cost', color='royalblue')
plt.plot(df['datetime'], equity_after - initial_capital, label='After 0.1% cost', color='crimson')
plt.title(f'Mean-Reversion Backtest (x={x}, SL={sl * 100:.1f}, TP={tp * 100:.1f}%)', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Now, we use `optuna` to optimize...

In [ ]:
# ====================== OPTUNA OBJECTIVE ======================
def objective(trial, df, closes, lows, initial_capital, n, fee_rate=0.001):
    x = trial.suggest_int('x', 1, 50)
    sl = trial.suggest_float('sl', 0.0001, 0.05)
    tp = trial.suggest_float('tp', 0.0001, 0.05)

    # signal (fast pandas)
    past_min = df['Low'].shift(1).rolling(window=x, min_periods=x).min().values
    signal = (lows < past_min) & (~np.isnan(past_min))

    equity, traded, _, _ = simulate(closes, signal, sl, tp, initial_capital, fee_rate)
    metrics = compute_metrics(equity, initial_capital, traded, n)
    return metrics['ann_sharpe']   # maximize after-cost Sharpe


# ====================== OPTUNA OPTIMIZATION (3 params) ======================
n_trials = 1_000
print(f"\nStarting Optuna optimization ({n_trials} trials) to maximize after-cost Sharpe...")
start_time = time.time()

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(lambda trial: objective(trial, df, closes, lows, initial_capital, n),
               n_trials=n_trials, show_progress_bar=True)

best_params = study.best_params
best_x = best_params['x']
best_sl = best_params['sl']
best_tp = best_params['tp']
best_sharpe = study.best_value

print(f"\nOPTUNA BEST: x={best_x}, SL={best_sl * 100:.2f}%, TP={best_tp * 100:.2f}% → Sharpe = {best_sharpe:.4f}")
print(f"Optimization finished in {time.time() - start_time:.1f}s")

# ====================== RUN BEST PARAMETERS & SAVE FOR DASHBOARD ======================
print("\nRunning best parameters for final equity curve...")
past_min_best = df['Low'].shift(1).rolling(window=best_x, min_periods=best_x).min().values
signal_best = (lows < past_min_best) & (~np.isnan(past_min_best))

equity_before_best, _, _, _ = simulate(closes, signal_best, best_sl, best_tp, initial_capital, 0.0)
equity_after_best, traded_best, max_lev_best, trades_best = simulate(closes, signal_best, best_sl, best_tp, initial_capital, 0.001)
metrics_best = compute_metrics(equity_after_best, initial_capital, traded_best, n)

# Save for Flask
equity_df = pd.DataFrame({
    'datetime': df['datetime'],
    'pnl_before': equity_before_best - initial_capital,
    'pnl_after': equity_after_best - initial_capital
})
equity_df.to_csv('equity_curve.csv', index=False)
print("✅ Saved 'equity_curve.csv' (BEST parameters) for dashboard.")

# Final metrics
print("\n" + "=" * 80)
print(f"BEST RESULTS (x={best_x}, SL={best_sl * 100:.2f}%, TP={best_tp * 100:.2f}%)")
print("=" * 80)
print(f"Final PnL AFTER  : ${metrics_best['final_pnl']:,.2f}")
print(f"Sharpe AFTER     : {metrics_best['sharpe']:.4f}")
print(f"Ann. Sharpe      : {metrics_best['ann_sharpe']:.2f}")
print(f"Max DD           : {metrics_best['max_dd_pct']:.2f}%")
print(f"Total trades     : {trades_best:,}")
print(f"Ann. Turnover    : {metrics_best['ann_turnover']:.2f}x")
print(f"Max Leverage     : {max_lev_best:.2f}x")
print("=" * 80)

# Quick plot of best
plt.figure(figsize=(14, 7))
plt.plot(df['datetime'], equity_before_best - initial_capital, label='Before cost', color='royalblue')
plt.plot(df['datetime'], equity_after_best - initial_capital, label='After cost', color='crimson')
plt.title(f'BEST Strategy (x={best_x}, SL={best_sl * 100:.2f}%, TP={best_tp * 100:.2f}%)', fontsize=16)
plt.xlabel('Time')
plt.ylabel('Cumulative PnL (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n✅ Optimization complete! equity_curve.csv is ready for the dashboard.")